# 04 — Modelo de Clasificación: Predicción de Riesgo de Retraso

**Tarea:** predecir `es_retraso` (0 = a tiempo, 1 = retraso)  
**Arquitectura:** igual que el modelo de regresión, solo cambia la capa de salida  
**Loss:** BinaryCrossentropy · **Manejo de desbalance:** class_weight

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight

print('TensorFlow:', tf.__version__)

DATA_DIR   = Path('../data/processed')
MODELS_DIR = Path('../models')
OUT_DIR    = Path('../outputs/graficas')

## 1. Cargar datos procesados

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
val   = pd.read_csv(DATA_DIR / 'val.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')

with open(DATA_DIR / 'preprocesadores.pkl', 'rb') as f:
    prep = pickle.load(f)

FEATURES_NUM = prep['features_num']
FEATURES_CAT = prep['features_cat']
CARDS        = prep['cardinalidades']

print(f'Tasa retraso en train: {train["es_retraso"].mean():.1%}')

## 2. Preparar inputs y calcular class_weight

In [ ]:
def preparar_inputs(df):
    X_num = df[FEATURES_NUM].values.astype('float32')
    X_cat = [df[col].values.astype('int32') for col in FEATURES_CAT]
    return [X_num] + X_cat

X_train = preparar_inputs(train)
y_train = train['es_retraso'].values.astype('float32')

X_val   = preparar_inputs(val)
y_val   = val['es_retraso'].values.astype('float32')

X_test  = preparar_inputs(test)
y_test  = test['es_retraso'].values.astype('float32')

# Calcular pesos de clase para compensar el desbalance
pesos_array = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_train
)
class_weight = {0: pesos_array[0], 1: pesos_array[1]}
print(f'Class weights: {class_weight}')

## 3. Arquitectura de la red

In [ ]:
def construir_modelo_clasificacion(cards, n_num):
    """
    Misma arquitectura base que regresión.
    Salida: sigmoid → probabilidad de retraso.
    """
    # Entrada numérica
    inp_num = keras.Input(shape=(n_num,), name='input_numerico')

    # Embeddigns categóricos
    emb_outputs = []
    emb_inputs  = []
    for col, card in cards.items():
        dim = min(50, (card // 2) + 1)
        inp = keras.Input(shape=(1,), name=f'input_{col}', dtype='int32')
        emb = layers.Embedding(input_dim=card, output_dim=dim, name=f'emb_{col}')(inp)
        emb = layers.Flatten()(emb)
        emb_inputs.append(inp)
        emb_outputs.append(emb)

    # Concatenar
    x = layers.Concatenate()([inp_num] + emb_outputs)

    # Bloques densos
    for unidades in [256, 128, 64]:
        x = layers.Dense(unidades, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3 if unidades == 256 else 0.2)(x)

    # Salida clasificación binaria
    salida = layers.Dense(1, activation='sigmoid', name='es_retraso')(x)

    modelo = keras.Model(
        inputs=[inp_num] + emb_inputs,
        outputs=salida,
        name='modelo_clasificacion'
    )
    return modelo

modelo = construir_modelo_clasificacion(CARDS, n_num=len(FEATURES_NUM))
modelo.summary()

## 4. Compilar y entrenar

In [ ]:
modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=10,
        restore_best_weights=True,
        mode='max', verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, verbose=1
    )
]

historia = modelo.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=256,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

## 5. Curva de aprendizaje

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(historia.history['loss'],     label='Train')
axes[0].plot(historia.history['val_loss'], label='Val')
axes[0].set_title('Loss (Binary Crossentropy)')
axes[0].set_xlabel('Época')
axes[0].legend()

axes[1].plot(historia.history['auc'],     label='Train')
axes[1].plot(historia.history['val_auc'], label='Val')
axes[1].set_title('AUC-ROC')
axes[1].set_xlabel('Época')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'curva_clasificacion.png', dpi=100)
plt.show()

## 6. Métricas en test

In [ ]:
from sklearn.metrics import (
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

y_prob = modelo.predict(X_test).flatten()
y_pred = (y_prob >= 0.5).astype(int)

print(f'AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['A tiempo', 'Retraso']))

## 7. Guardar modelo

In [ ]:
modelo.save(MODELS_DIR / 'modelo_clasificacion.keras')
print('Modelo guardado en models/modelo_clasificacion.keras')